In [ ]:
# ============================================================
# NSE DAILY BHAVCOPY DOWNLOADER
#
# Purpose:
#   Maintain a complete NSE CM daily dataset from START_DATE
#   through the latest available date.
#
# Storage:
#   /content/drive/MyDrive/quant/data/parquet/
#
# Structure:
#   parquet/
#       year=2011/
#           nse_cm_20110103.parquet
#           nse_cm_20110104.parquet
#           ...
#       year=2012/
#       ...
#
# IMPORTANT:
#   - Existing VALID files are never downloaded again.
#   - Missing dates are downloaded.
#   - Corrupt/invalid files are redownloaded.
#   - Existing files with NaT/incorrect dates are repaired.
#   - Weekends are skipped.
#   - NSE holidays simply return 404 and are recorded as
#     "not_found".
#   - Running this script repeatedly is safe.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import io
import random
import re
import time
import zipfile

from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm


# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# YOUR ACTUAL QUANT PROJECT ON GOOGLE DRIVE
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/quant"
)

DATA_DIR = BASE_DIR / "data"

RAW_DIR = DATA_DIR / "raw"

PARQUET_DIR = DATA_DIR / "parquet"

MANIFEST_FILE = (
    DATA_DIR / "download_manifest.csv"
)


# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------

START_DATE = date(
    2011,
    1,
    1
)

# None = automatically use today's date
END_DATE = None


# ------------------------------------------------------------
# STOCK FILTER
# ------------------------------------------------------------
#
# None = download ALL securities
#
# Keep this None for the historical dataset.
# ------------------------------------------------------------

STOCKS = None


# ------------------------------------------------------------
# REQUEST SETTINGS
# ------------------------------------------------------------

REQUEST_DELAY_MIN = 0.5

REQUEST_DELAY_MAX = 1.0

MAX_RETRIES = 6

TIMEOUT = 45


USER_AGENT = (
    "Mozilla/5.0 "
    "(Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 "
    "(KHTML, like Gecko) "
    "Chrome/139.0.0.0 "
    "Safari/537.36"
)


# ============================================================
# 3. DIRECTORY SETUP
# ============================================================

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PARQUET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. NSE URL BUILDERS
# ============================================================

def legacy_url(d: date) -> str:

    """
    Legacy CM Bhavcopy.

    Used through 05-Jul-2024.
    """

    return (
        "https://nsearchives.nseindia.com/content/"
        f"historical/EQUITIES/{d.year}/"
        f"{d.strftime('%b').upper()}/"
        f"cm{d.strftime('%d%b%Y').upper()}"
        "bhav.csv.zip"
    )


def udiff_url(d: date) -> str:

    """
    UDiFF CM Common Bhavcopy.

    Used from 08-Jul-2024 onward.
    """

    return (
        "https://nsearchives.nseindia.com/content/cm/"
        "BhavCopy_NSE_CM_0_0_0_"
        f"{d.strftime('%Y%m%d')}"
        "_F_0000.csv.zip"
    )


def get_url(d: date) -> str:

    if d <= date(2024, 7, 5):

        return legacy_url(d)

    return udiff_url(d)


# ============================================================
# 5. HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({

    "User-Agent":
        USER_AGENT,

    "Accept":
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,"
            "image/avif,image/webp,*/*;q=0.8"
        ),

    "Accept-Language":
        "en-US,en;q=0.9",

    "Connection":
        "keep-alive",

    "Referer":
        "https://www.nseindia.com/",

    "Origin":
        "https://www.nseindia.com",

})


def initialize_nse_session():

    print(
        "Initializing NSE session..."
    )

    try:

        response = session.get(
            "https://www.nseindia.com/",
            timeout=TIMEOUT
        )

        print(
            "NSE homepage:",
            response.status_code
        )

    except Exception as exc:

        print(
            "Warning: NSE homepage request failed:",
            exc
        )

    time.sleep(1)


initialize_nse_session()


# ============================================================
# 6. MANIFEST
# ============================================================

MANIFEST_COLUMNS = [
    "date",
    "url",
    "status",
    "rows",
    "file_type",
    "error",
    "timestamp",
]


def load_manifest():

    if not MANIFEST_FILE.exists():

        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )

    try:

        df = pd.read_csv(
            MANIFEST_FILE,
            dtype={
                "date": str,
                "status": str,
            }
        )

        return df

    except Exception as exc:

        print(
            "Warning: Could not read manifest:",
            exc
        )

        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )


manifest = load_manifest()


def save_manifest():

    tmp = MANIFEST_FILE.with_suffix(
        ".tmp.csv"
    )

    manifest.to_csv(
        tmp,
        index=False
    )

    tmp.replace(
        MANIFEST_FILE
    )


def record_manifest(
    d,
    url,
    status,
    rows=0,
    file_type="",
    error=""
):

    global manifest

    record = pd.DataFrame([{

        "date":
            d.isoformat(),

        "url":
            url,

        "status":
            status,

        "rows":
            rows,

        "file_type":
            file_type,

        "error":
            error,

        "timestamp":
            datetime.now().isoformat(),

    }])

    manifest = manifest[
        manifest["date"] != d.isoformat()
    ]

    manifest = pd.concat(
        [
            manifest,
            record
        ],
        ignore_index=True
    )

    save_manifest()


# ============================================================
# 7. PARQUET PATH
# ============================================================

def parquet_path(
    d: date
) -> Path:

    year_dir = (
        PARQUET_DIR /
        f"year={d.year}"
    )

    year_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    return (
        year_dir /
        f"nse_cm_{d:%Y%m%d}.parquet"
    )


# ============================================================
# 8. VALIDATE EXISTING FILE
# ============================================================

def validate_existing_parquet(
    path: Path,
    expected_date: date
):

    """
    Return:

        valid
        repaired
        invalid
    """

    expected_timestamp = pd.Timestamp(
        expected_date
    )

    try:

        # ----------------------------------------------------
        # File must have non-zero size
        # ----------------------------------------------------

        if path.stat().st_size < 1000:

            return "invalid"


        # ----------------------------------------------------
        # Read Parquet
        # ----------------------------------------------------

        df = pd.read_parquet(
            path
        )


        # ----------------------------------------------------
        # Empty
        # ----------------------------------------------------

        if df.empty:

            return "invalid"


        # ----------------------------------------------------
        # Required columns
        # ----------------------------------------------------

        required = {
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume",
        }

        missing = (
            required
            - set(df.columns)
        )

        if missing:

            return "invalid"


        # ----------------------------------------------------
        # Check date
        # ----------------------------------------------------

        dates = pd.to_datetime(
            df["date"],
            errors="coerce"
        )


        # ----------------------------------------------------
        # Correct
        # ----------------------------------------------------

        if (
            dates.notna().all()
            and
            (dates == expected_timestamp).all()
        ):

            return "valid"


        # ----------------------------------------------------
        # Repair
        # ----------------------------------------------------

        print(
            f"Repairing date column: "
            f"{path.name}"
        )

        df["date"] = expected_timestamp

        tmp_path = path.with_suffix(
            ".repair.tmp.parquet"
        )

        df.to_parquet(
            tmp_path,
            engine="pyarrow",
            compression="zstd",
            index=False
        )

        tmp_path.replace(
            path
        )

        return "repaired"


    except Exception as exc:

        print(
            f"Invalid Parquet "
            f"{path.name}: {exc}"
        )

        return "invalid"


# ============================================================
# 9. DOWNLOAD ZIP
# ============================================================

def download_file(
    d: date
):

    url = get_url(d)

    last_error = ""

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = session.get(
                url,
                timeout=TIMEOUT
            )

            status = (
                response.status_code
            )


            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if status == 200:

                content = response.content


                # Basic sanity check
                if len(content) < 1000:

                    last_error = (
                        "HTTP 200 but "
                        "response is too small"
                    )

                    time.sleep(
                        1 + random.random()
                    )

                    continue


                # Validate ZIP
                try:

                    with zipfile.ZipFile(
                        io.BytesIO(content)
                    ) as z:

                        bad_file = z.testzip()

                        if bad_file is not None:

                            raise ValueError(
                                f"Corrupt ZIP member: "
                                f"{bad_file}"
                            )

                except zipfile.BadZipFile:

                    last_error = (
                        "Response is not a valid ZIP"
                    )

                    time.sleep(
                        1 + random.random()
                    )

                    continue


                return content


            # ------------------------------------------------
            # 404
            # ------------------------------------------------

            if status == 404:

                return None


            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            if status == 429:

                wait = min(
                    60,
                    2 ** attempt
                )

                wait += random.uniform(
                    0,
                    2
                )

                print(
                    f"\n429 rate limit. "
                    f"Sleeping {wait:.1f}s..."
                )

                time.sleep(
                    wait
                )

                continue


            # ------------------------------------------------
            # ACCESS / SESSION
            # ------------------------------------------------

            if status in (
                401,
                403
            ):

                last_error = (
                    f"HTTP {status}"
                )

                print(
                    f"\nHTTP {status}. "
                    "Refreshing NSE session..."
                )

                initialize_nse_session()

                time.sleep(
                    min(
                        30,
                        2 ** attempt
                    )
                )

                continue


            # ------------------------------------------------
            # SERVER ERROR
            # ------------------------------------------------

            if status >= 500:

                last_error = (
                    f"HTTP {status}"
                )

                wait = min(
                    60,
                    2 ** attempt
                )

                wait += random.uniform(
                    0,
                    2
                )

                time.sleep(
                    wait
                )

                continue


            last_error = (
                f"Unexpected HTTP {status}"
            )


        except requests.RequestException as exc:

            last_error = str(exc)

            wait = min(
                60,
                2 ** attempt
            )

            wait += random.uniform(
                0,
                2
            )

            time.sleep(
                wait
            )


    raise RuntimeError(
        f"Failed downloading {url} "
        f"after {MAX_RETRIES} attempts. "
        f"Last error: {last_error}"
    )


# ============================================================
# 10. READ ZIP
# ============================================================

def read_zip_csv(
    content
):

    with zipfile.ZipFile(
        io.BytesIO(content)
    ) as z:

        csv_files = [
            f
            for f in z.namelist()
            if f.lower().endswith(".csv")
        ]

        if not csv_files:

            raise ValueError(
                "No CSV file found in ZIP"
            )

        filename = csv_files[0]

        with z.open(filename) as f:

            df = pd.read_csv(
                f,
                low_memory=False
            )

    return df, filename


# ============================================================
# 11. COLUMN NORMALIZATION
# ============================================================

def normalize_column_name(
    column
):

    column = str(column).strip()

    column = column.replace(
        "\ufeff",
        ""
    )

    column = re.sub(
        r"\s+",
        "_",
        column
    )

    column = re.sub(
        r"[^A-Za-z0-9_]",
        "",
        column
    )

    return column.upper()


def normalize_columns(
    df
):

    df = df.copy()

    df.columns = [
        normalize_column_name(c)
        for c in df.columns
    ]

    return df


def find_column(
    df,
    candidates
):

    for column in candidates:

        if column in df.columns:

            return column

    return None


# ============================================================
# 12. NORMALIZE BHAVCOPY
# ============================================================

def normalize_bhavcopy(
    df,
    trading_date
):

    df = normalize_columns(
        df
    )


    # --------------------------------------------------------
    # Identify columns
    # --------------------------------------------------------

    symbol_col = find_column(
        df,
        [
            "SYMBOL",
            "TCKRSYMB",
            "TICKER",
        ]
    )

    series_col = find_column(
        df,
        [
            "SERIES",
            "SCTYSRS",
        ]
    )

    isin_col = find_column(
        df,
        [
            "ISIN",
            "ISINNUMBER",
            "ISINNO",
        ]
    )

    open_col = find_column(
        df,
        [
            "OPEN",
            "OPNPRIC",
        ]
    )

    high_col = find_column(
        df,
        [
            "HIGH",
            "HGHPRIC",
        ]
    )

    low_col = find_column(
        df,
        [
            "LOW",
            "LWPRIC",
        ]
    )

    close_col = find_column(
        df,
        [
            "CLOSE",
            "CLSPRIC",
        ]
    )

    prev_close_col = find_column(
        df,
        [
            "PREVCLOSE",
            "PRVCLSGPRIC",
        ]
    )

    volume_col = find_column(
        df,
        [
            "TOTTRDQTY",
            "TOTTRADQTY",
            "TOTTRDQVOL",
            "TOTTRDQUANTITY",
        ]
    )

    turnover_col = find_column(
        df,
        [
            "TOTTRDVAL",
            "TOTTRDVALUE",
        ]
    )

    trades_col = find_column(
        df,
        [
            "TOTALTRADES",
            "TOTTRDNUM",
            "NOOFTDR",
        ]
    )


    # --------------------------------------------------------
    # Required fields
    # --------------------------------------------------------

    if symbol_col is None:

        raise ValueError(
            "Could not identify SYMBOL column. "
            f"Columns: {list(df.columns)}"
        )

    if close_col is None:

        raise ValueError(
            "Could not identify CLOSE column. "
            f"Columns: {list(df.columns)}"
        )


    # --------------------------------------------------------
    # Build output
    # --------------------------------------------------------

    expected_timestamp = pd.Timestamp(
        trading_date
    )

    out = pd.DataFrame(
        index=df.index
    )

    out["date"] = expected_timestamp

    out["symbol"] = (
        df[symbol_col]
        .astype("string")
        .str.strip()
    )


    # --------------------------------------------------------
    # Series
    # --------------------------------------------------------

    if series_col:

        out["series"] = (
            df[series_col]
            .astype("string")
            .str.strip()
        )

    else:

        out["series"] = pd.Series(
            pd.NA,
            index=df.index,
            dtype="string"
        )


    # --------------------------------------------------------
    # ISIN
    # --------------------------------------------------------

    if isin_col:

        out["isin"] = (
            df[isin_col]
            .astype("string")
            .str.strip()
        )

    else:

        out["isin"] = pd.Series(
            pd.NA,
            index=df.index,
            dtype="string"
        )


    # --------------------------------------------------------
    # Numeric helper
    # --------------------------------------------------------

    def numeric(
        column
    ):

        if column is None:

            return pd.Series(
                pd.NA,
                index=df.index,
                dtype="Float64"
            )

        return pd.to_numeric(
            df[column],
            errors="coerce"
        )


    out["open"] = numeric(
        open_col
    )

    out["high"] = numeric(
        high_col
    )

    out["low"] = numeric(
        low_col
    )

    out["close"] = numeric(
        close_col
    )

    out["prev_close"] = numeric(
        prev_close_col
    )

    out["volume"] = numeric(
        volume_col
    )

    out["turnover"] = numeric(
        turnover_col
    )

    out["trades"] = numeric(
        trades_col
    )


    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------

    out = out[
        out["symbol"].notna()
        &
        (out["symbol"] != "")
        &
        out["close"].notna()
    ]


    # --------------------------------------------------------
    # Equity series
    # --------------------------------------------------------

    if out["series"].notna().any():

        out = out[
            out["series"].isin([
                "EQ",
                "BE",
                "BZ",
                "SM",
                "ST",
                "SZ",
            ])
        ]


    # --------------------------------------------------------
    # Optional stock filter
    # --------------------------------------------------------

    if STOCKS is not None:

        out = out[
            out["symbol"].isin(
                STOCKS
            )
        ]


    # --------------------------------------------------------
    # Final date assignment
    # --------------------------------------------------------

    out["date"] = expected_timestamp


    return out.reset_index(
        drop=True
    )


# ============================================================
# 13. WRITE PARQUET SAFELY
# ============================================================

def write_parquet(
    df,
    trading_date
):

    path = parquet_path(
        trading_date
    )

    tmp_path = path.with_suffix(
        ".tmp.parquet"
    )


    df.to_parquet(
        tmp_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )


    # Atomic replacement
    tmp_path.replace(
        path
    )


    return path


# ============================================================
# 14. PROCESS ONE DATE
# ============================================================

def process_date(
    trading_date
):

    path = parquet_path(
        trading_date
    )

    url = get_url(
        trading_date
    )


    # ========================================================
    # EXISTING FILE
    # ========================================================

    if path.exists():

        result = (
            validate_existing_parquet(
                path,
                trading_date
            )
        )


        if result == "valid":

            record_manifest(
                trading_date,
                url,
                "skipped_existing"
            )

            return "skipped_existing"


        if result == "repaired":

            record_manifest(
                trading_date,
                url,
                "repaired_existing"
            )

            return "repaired_existing"


        # Invalid file
        print(
            f"\nDeleting invalid file: "
            f"{path}"
        )

        try:

            path.unlink()

        except Exception:

            pass


    # ========================================================
    # DOWNLOAD
    # ========================================================

    try:

        content = download_file(
            trading_date
        )


        # ----------------------------------------------------
        # 404
        # ----------------------------------------------------

        if content is None:

            record_manifest(
                trading_date,
                url,
                "not_found"
            )

            return "not_found"


        # ----------------------------------------------------
        # Read ZIP
        # ----------------------------------------------------

        raw_df, filename = (
            read_zip_csv(
                content
            )
        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        df = normalize_bhavcopy(
            raw_df,
            trading_date
        )


        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        if df.empty:

            raise ValueError(
                "Normalized dataset is empty"
            )


        if df["date"].isna().any():

            raise ValueError(
                "Date contains NaT"
            )


        # ----------------------------------------------------
        # Write
        # ----------------------------------------------------

        output = write_parquet(
            df,
            trading_date
        )


        # ----------------------------------------------------
        # Verify what we wrote
        # ----------------------------------------------------

        validation = (
            validate_existing_parquet(
                output,
                trading_date
            )
        )


        if validation != "valid":

            raise ValueError(
                f"Post-write validation failed: "
                f"{validation}"
            )


        # ----------------------------------------------------
        # Manifest
        # ----------------------------------------------------

        record_manifest(
            trading_date,
            url,
            "success",
            rows=len(df),
            file_type=filename
        )


        # ----------------------------------------------------
        # Delay
        # ----------------------------------------------------

        time.sleep(
            random.uniform(
                REQUEST_DELAY_MIN,
                REQUEST_DELAY_MAX
            )
        )


        return "success"


    except Exception as exc:

        record_manifest(
            trading_date,
            url,
            "error",
            error=str(exc)
        )

        return "error"


# ============================================================
# 15. DETERMINE DATE RANGE
# ============================================================

if END_DATE is None:

    END_DATE = date.today()


all_dates = []

current = START_DATE


while current <= END_DATE:

    # Only weekdays.
    # NSE holidays will naturally return 404.
    if current.weekday() < 5:

        all_dates.append(
            current
        )

    current += timedelta(
        days=1
    )


# ============================================================
# 16. DISPLAY CONFIGURATION
# ============================================================

print()
print("=" * 72)
print("NSE BHAVCOPY DOWNLOADER")
print("=" * 72)

print(
    f"Start date : {START_DATE}"
)

print(
    f"End date   : {END_DATE}"
)

print(
    f"Weekdays   : {len(all_dates):,}"
)

print(
    f"Output     : {PARQUET_DIR}"
)

print(
    f"Manifest   : {MANIFEST_FILE}"
)

print("=" * 72)
print()


# ============================================================
# 17. PROCESS ALL DATES
# ============================================================

results = {
    "success": 0,
    "skipped_existing": 0,
    "repaired_existing": 0,
    "not_found": 0,
    "error": 0,
}


for trading_date in tqdm(
    all_dates,
    desc="NSE Bhavcopy"
):

    result = process_date(
        trading_date
    )

    results[result] += 1


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print()
print("=" * 72)
print("DOWNLOAD COMPLETE")
print("=" * 72)

for key, value in results.items():

    print(
        f"{key:24s}: {value:,}"
    )

print()
print(
    "Parquet:",
    PARQUET_DIR
)

print(
    "Manifest:",
    MANIFEST_FILE
)

print("=" * 72)